# AI Docking with gnina — CNN-scored docking of FA19 into COX-2
### The deep-learning upgrade to your Vina workflow

You've docked FA19 with AutoDock Vina. **gnina** does the same job but replaces the scoring step with a **3D convolutional neural network (CNN)** trained on thousands of real protein–ligand complexes. So:

| | AutoDock Vina | gnina |
|---|---|---|
| Search (pose sampling) | Monte-Carlo + local opt | **same engine** (gnina is built on Vina/smina) |
| Scoring (rank poses) | hand-crafted empirical formula (kcal/mol) | **a learned 3D CNN** |
| What you get out | one affinity (kcal/mol) | **CNNscore** + **CNNaffinity** + the empirical affinity |

gnina gives you **two AI numbers per pose**:
- **CNNscore** (0–1): the CNN's confidence that the pose is a *correct, realistic binding pose*. Use it to judge "do I believe this pose?"
- **CNNaffinity** (in pK units; higher = tighter): the CNN's predicted binding strength. Use it to *rank* molecules.
- It still also reports **minimizedAffinity** (the Vina/Vinardo empirical kcal/mol), so you can compare the AI and the physics side by side.

> **Run this in Colab with a GPU runtime** (Runtime → Change runtime type → GPU). The CNN runs on CPU too, but slowly. I built this carefully but could not execute it here — run the install cell and check `gnina --version` prints before continuing.


## 1 · Install gnina
gnina ships as a **single precompiled binary** on GitHub — no build needed. We download it, make it executable, and add Open Babel (receptor/ligand prep) and RDKit.


In [ ]:
# precompiled gnina binary (GPU recommended)
!wget -q https://github.com/gnina/gnina/releases/latest/download/gnina -O /usr/local/bin/gnina
!chmod +x /usr/local/bin/gnina
!apt-get -qq install -y openbabel libgomp1 >/dev/null 2>&1
!pip -q install rdkit py3Dmol requests numpy 2>/dev/null
# sanity check — must print a version. If it errors on a missing lib, see the note at the bottom.
!gnina --version | head -3

## 2 · Prepare the receptor (COX-2, PDB 3LN1)
Same idea as in Vina: download the crystal, keep one protein chain, drop water/ligand/ions, protonate, and write a PDBQT. We also pull out the bound **celecoxib** — gnina will use it to place the search box automatically (its `--autobox_ligand` feature, easier than typing box coordinates).


In [ ]:
import requests
pdb = requests.get("https://files.rcsb.org/download/3LN1.pdb", timeout=60).text
open("3LN1.pdb","w").write(pdb)

with open("receptor_clean.pdb","w") as f:
    for l in pdb.splitlines():
        if l.startswith("ATOM") and l[21]=="A": f.write(l+"\n")
    f.write("TER\nEND\n")
!obabel receptor_clean.pdb -O receptor.pdbqt -xr -p 7.4 2>/dev/null

# reference ligand (crystal celecoxib) -> defines the box
with open("cel_crystal.pdb","w") as f:
    for l in pdb.splitlines():
        if l.startswith("HETATM") and l[17:20].strip()=="CEL" and l[21]=="A": f.write(l+"\n")
    f.write("END\n")
import os; print("receptor.pdbqt:", os.path.getsize("receptor.pdbqt"), "bytes  | reference ligand ready")

## 3 · Prepare the ligand (FA19)
gnina is flexible about input (it uses Open Babel internally), so a clean 3D **SDF** from RDKit is ideal.


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
m = Chem.AddHs(Chem.MolFromSmiles("COc1ccc(C(=O)Oc2ccc(/C=C/C(=O)O)cc2OC)cc1"))
AllChem.EmbedMolecule(m, randomSeed=42); AllChem.MMFFOptimizeMolecule(m)
Chem.MolToMolFile(m, "fa19.sdf")
print("fa19.sdf written")

## 4 · Dock with the CNN
Key flags:
- **`--autobox_ligand cel_crystal.pdb`** — build the search box around the known binder (+4 Å padding by default). No manual coordinates.
- **`--cnn_scoring rescore`** — dock with the fast empirical engine, then **CNN-rescore** the final poses. This is the practical default. Other modes: `all` (CNN used *during* search + refinement — the fullest "AI" mode, much slower, GPU strongly advised), `refinement`, or `none` (pure Vina-style).
- **`--seed 42`** — reproducibility (the search is stochastic).
- **`--num_modes 9`** — return 9 ranked poses.

When CNN scoring is on, gnina **ranks the output poses by CNNscore**, so pose 1 is the CNN's most-believed pose.


In [ ]:
!gnina -r receptor.pdbqt -l fa19.sdf --autobox_ligand cel_crystal.pdb \
       -o fa19_gnina.sdf.gz --cnn_scoring rescore --seed 42 --num_modes 9
print("\ndocking done -> fa19_gnina.sdf.gz")

## 5 · Read the AI scores
We read the output SDF and pull the three numbers per pose. **How to interpret:**
- **CNNscore** near 1.0 → the CNN strongly believes this is a real binding pose. Below ~0.5 → treat the pose with suspicion.
- **CNNaffinity** higher → stronger predicted binding (pK units; e.g. 6 ≈ 1 µM, 9 ≈ 1 nM). This is the number to *compare across molecules*.
- **minimizedAffinity** → the familiar empirical kcal/mol (more negative = better), directly comparable to your Vina number.


In [ ]:
import gzip
from rdkit import Chem
rows=[]
for mol in Chem.ForwardSDMolSupplier(gzip.open("fa19_gnina.sdf.gz")):
    if mol is None: continue
    g=lambda k: float(mol.GetProp(k)) if mol.HasProp(k) else None
    rows.append((g("CNNscore"), g("CNNaffinity"), g("minimizedAffinity")))
print("Pose | CNNscore | CNNaffinity (pK) | minimizedAffinity (kcal/mol)")
for i,(cs,ca,ma) in enumerate(rows,1):
    print(f"  {i:>2} |   {cs:.3f}  |     {ca:.2f}        |      {ma:.2f}")
best=rows[0]
print(f"\nTop pose: CNNscore {best[0]:.3f}, CNNaffinity {best[1]:.2f} pK, empirical {best[2]:.2f} kcal/mol")

## 6 · Validate — re-dock celecoxib (the trust test, AI edition)
Same principle as with Vina: re-dock the known ligand and check you recover its crystal pose (**RMSD < 2 Å**). With gnina you get a bonus signal — a high **CNNscore** on the redocked pose is independent evidence the CNN recognises a correct binding mode.


In [ ]:
!obabel cel_crystal.pdb -O cel.sdf 2>/dev/null
!gnina -r receptor.pdbqt -l cel.sdf --autobox_ligand cel_crystal.pdb \
       -o cel_redock.sdf.gz --cnn_scoring rescore --seed 42 --num_modes 5
# top redocked pose -> CNN score + RMSD to crystal
top=next(Chem.ForwardSDMolSupplier(gzip.open("cel_redock.sdf.gz")))
print("Redocked celecoxib CNNscore:", round(float(top.GetProp("CNNscore")),3))
!obabel cel_redock.sdf.gz -O cel_redock_top.sdf -f 1 -l 1 2>/dev/null
print("Pose RMSD vs crystal (want < 2.0 A):")
!obrms cel_crystal.pdb cel_redock_top.sdf

## 7 · See the pose

In [ ]:
import py3Dmol, gzip
v = py3Dmol.view(width=750, height=520)
v.addModel(open("receptor_clean.pdb").read(), "pdb")
v.setStyle({"cartoon": {"color": "spectrum"}})
v.addModel(gzip.open("fa19_gnina.sdf.gz").read().decode(), "sdf")
v.setStyle({"model": -1}, {"stick": {"colorscheme": "greenCarbon"}})
v.zoomTo({"model": -1}); v.show()

## 8 · gnina vs Vina vs MOE — what the comparison means
- **minimizedAffinity (gnina)** is Vina-family physics — compare it directly to your Vina kcal/mol.
- **CNNaffinity / CNNscore** are a *separate, learned* opinion. They are **not** in kcal/mol and shouldn't be compared numerically to MOE's −8.3; they answer a different question ("does a network trained on real complexes think this pose is good and tight?").
- The strong result is **convergence**: if MOE, Vina, *and* gnina's CNN all favour FA19 binding in the same pocket, that's three methods — two physics, one AI — agreeing. That's a far better portfolio claim than any single score.

## Honest limits
- **Applicability domain.** The CNN was trained on PDBbind/CrossDocked complexes; it's strongest on common drug-like chemotypes and protein families. Off-distribution systems get less reliable scores — treat CNNaffinity as a ranking aid, not truth.
- **GPU.** CNN scoring is slow on CPU; use a Colab GPU runtime.
- **CNNs can be overconfident.** Always keep the re-dock control: a high CNNscore on a pose that *fails* the celecoxib RMSD test means the setup, not the score, is the problem.
- **Keep Vina too.** gnina doesn't replace your Vina notebook — having both (empirical + AI) is the point.

## If the binary won't run
The precompiled `gnina` occasionally complains about a missing CUDA/Open Babel library. Fallbacks: install via conda (`conda install -c conda-forge gnina`), or use the official gnina **Docker** image, or run with `--cnn_scoring none` to confirm the docking pipeline works before turning the CNN on.
